## Install the package

First, clone the repo. In your terminal:

```bash
git clone https://github.com/EPPI-Centre/Flowde.git
cd Flowde
```

Install the package. In your terminal:

```bash
uv venv .venv --python 3.11
source .venv/bin/activate

uv pip install ipykernel jupyter ipywidgets
uv pip install -e ".[all]"

python -m ipykernel install --user --name flowde-env --display-name "Python (flowde-env)"
```

Then go into your notebook and select the `Python (flowde-env)` kernel.


## Download the Data

### Install git lfs

Check Git LFS is available. In your terminal:

```bash
git lfs version
```

If that command is not found, install Git LFS first. For windows, run the
following in your terminal:

```bash
winget install GitHub.GitLFS
```

### Install the data

Run this in your terminal in the `Flowde` directory:

```bash
git lfs install
git lfs pull --include="data/**"
```


## Extract Imgs


In [ ]:
from pathlib import Path

from flowde.extract_fns.paddle_layout_detect_extraction import (
    make_paddle_layout_extract_fn,
)
from flowde.extract_imgs import extract_imgs

PDF_DIR = Path("./../data/training-smoking-cessation/pdfs/")
EXTRACTED_IMGS_SAVE_DIR = Path(
    "./../data/training-smoking-cessation/extraction/my-extracted-images/"
)
N_JOBS = 1

In [ ]:
extract_fn = make_paddle_layout_extract_fn(device="cpu")

# Use gpu if you have installed gpu support:
# extract_fn = make_paddle_layout_extract_fn(device="gpu")

extract_imgs(
    pdf_dir=PDF_DIR,
    save_dir=EXTRACTED_IMGS_SAVE_DIR,
    extract_fn=extract_fn,
    n_jobs=N_JOBS,
)

## Classify Images


### Add a `.env`

At the root of your cloned repo, or in the same dir as this notebook, create a
`.env` file and add your OpenAI api key in the following format:

```text

OPENAI_API_KEY=your-openai-api-key

```


### Run classification


In [ ]:
from pathlib import Path

from flowde.classify_fns.openai_classify_fn import make_openai_classify_fn
from flowde.classify_imgs import classify_imgs

IMGS_TO_CLASSIFY_DIR = EXTRACTED_IMGS_SAVE_DIR
MODEL = "gpt-5.4-mini"  # Good performance but cheap
JSON_RESULTS_PATH = Path(
    "../data/training-smoking-cessation/classification/my-classifications.json"
)
POSITIVE_IMG_SAVE_DIR = Path(
    "../data/training-smoking-cessation/classification/positive-images/"
)

# MODEL = "gpt-5.5"  # Near perfect performance, but more expensive
MODEL_EFFORT = "high"
INPUT_TEXT = """
Classify whether this image is a flowchart.

Return 1 if it is a flowchart.
Return 0 if it is not a flowchart.
"""

In [ ]:
classify_fn = make_openai_classify_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
)


labels = classify_imgs(
    classify_fn=classify_fn,
    img_dir=IMGS_TO_CLASSIFY_DIR,
    json_path=JSON_RESULTS_PATH,
    positive_img_save_dir=POSITIVE_IMG_SAVE_DIR,
    positive_classes={1},
)

## Rotate Images


In [ ]:
from pathlib import Path

from pydantic import BaseModel

from flowde.classify_fns.classify_types import RotationLabel
from flowde.classify_fns.openai_classify_fn import make_openai_classify_fn
from flowde.rotate_imgs import rotate_imgs

IMGS_TO_ROTATE_DIR = POSITIVE_IMG_SAVE_DIR
ROTATED_IMGS_SAVE_DIR = Path(
    "../data/training-smoking-cessation/rotation/rotation-corrected-images/"
)
JSON_ROTATIONS_RESULTS_PATH = Path(
    "../data/training-smoking-cessation/rotation/rotation-labels.json"
)

MODEL = "gpt-5.4-mini"  # Good performance but cheap
# MODEL = "gpt-5.5"  # Near perfect performance, but more expensive
MODEL_EFFORT = "high"
INPUT_TEXT = """
Return the clockwise angle required to correctly orient the image,
such that the majority of text reads left to right, top to bottom.
"""

In [ ]:
class RotationClassification(BaseModel):
    label: RotationLabel  # RotationLabel = Literal[0, 90, 180, 270]


classify_fn = make_openai_classify_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    result_structure=RotationClassification,
    effort=MODEL_EFFORT,
)

rotation_labels = rotate_imgs(
    classify_fn=classify_fn,
    img_dir=IMGS_TO_ROTATE_DIR,
    save_dir=ROTATED_IMGS_SAVE_DIR,
    json_path=JSON_ROTATIONS_RESULTS_PATH,
)

## Parse Images

For this step, if you want to get good results, it's important to use one of the
intelligent expensive models like `gpt5.5` and to parse the CONSORT diagrams in
parts


### Parse Nodes


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

IMGS_TO_PARSE_DIR = ROTATED_IMGS_SAVE_DIR
PARSED_NODES_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/nodes")

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse nodes
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"node_text"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    save_dir=PARSED_NODES_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

### Parse Labels


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

IMGS_TO_PARSE_DIR = ROTATED_IMGS_SAVE_DIR
PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/labels")

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse labels
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"labels"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    save_dir=PARSED_LABELS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

### Parse Flow


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

IMGS_TO_PARSE_DIR = ROTATED_IMGS_SAVE_DIR
PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_DIR = PARSED_LABELS_SAVE_DIR
PARSED_FLOWS_SAVE_DIR = Path("../data/training-smoking-cessation/parsing/pred/flows")

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse flows
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"flow"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    labels_dir=PARSED_LABELS_DIR,
    save_dir=PARSED_FLOWS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)

### Parse Additional Texts and combine into full CONSORT json


In [ ]:
from pathlib import Path

from flowde.parse_imgs import parse_imgs
from flowde.parsing_fns.openai_parse import make_openai_parse_fn

IMGS_TO_PARSE_DIR = ROTATED_IMGS_SAVE_DIR
PARSED_NODES_DIR = PARSED_NODES_SAVE_DIR
PARSED_LABELS_DIR = PARSED_LABELS_SAVE_DIR
PARSED_FLOWS_DIR = PARSED_FLOWS_SAVE_DIR
PARSED_FLOWCHARTS_SAVE_DIR = Path(
    "../data/training-smoking-cessation/parsing/pred/full-flowcharts"
)

MODEL = "gpt-5.4-mini"  # bad performance but cheap
# MODEL = "gpt-5.5"  # good performance but more expensive
MODEL_EFFORT = "high"
IMG_RANGE_TO_PARSE = (0, 1000)
INPUT_TEXT = """
Parse additional text
"""

In [ ]:
parse_fn = make_openai_parse_fn(
    input_text=INPUT_TEXT,
    model=MODEL,
    effort=MODEL_EFFORT,
    parts_to_parse={"node_text", "labels", "flow", "additional_texts"},
)


responses = parse_imgs(
    parse_fn=parse_fn,
    img_dir=IMGS_TO_PARSE_DIR,
    nodes_dir=PARSED_NODES_DIR,
    labels_dir=PARSED_LABELS_DIR,
    flow_dir=PARSED_FLOWS_DIR,
    save_dir=PARSED_FLOWCHARTS_SAVE_DIR,
    range_indices=IMG_RANGE_TO_PARSE,
)